# StoryForge AI — ShopFlow POC v3

> *From a plain English user story to executable test scripts — fully on-premise, zero data egress, runs entirely on your infrastructure.*

---

## User Story — US-001

**As a registered shopper on ShopFlow**, I want to search for a product, add it to my cart, and complete payment so that my **order is placed** and I receive an **order confirmation with a unique order ID**.

---

## Pipeline

| Stage | Description | Output |
|-------|-------------|--------|
| 1 | **Parsing Engine** — Mistral 7B extracts actor, actions, AC, business rules | Structured JSON (ParsedSpec) |
| 2 | **Test Case Generator** — LangChain 4-agent loop generates test cases | Test Case JSON |
| 3 | **Test Data Generator** — Faker (seed=42) generates deterministic data | test_data.json |
| 4 | **Coverage Report** — maps every test case to acceptance criteria | AC Coverage PASS/FAIL |
| 5 | **Code Synthesis** — Jinja2 renders executable test scripts | .java / .spec.ts files |

---

## How to Run

```bash
# 1. Install Ollama and pull Mistral
ollama pull mistral

# 2. Install dependencies
pip install pydantic jinja2 pyyaml faker rich requests

# 3. Open this notebook and run Kernel → Restart and Run All
```

> **Note:** In this POC the LLM parsing stage is simulated with a pre-validated ParsedSpec to allow the notebook to run without Ollama. Replace `SIMULATE_LLM = True` with `False` to use live Mistral inference.

In [1]:
# ── Dependencies ──────────────────────────────────────────────
import json, yaml, os, requests
from faker import Faker
from jinja2 import Environment, FileSystemLoader
from pydantic import BaseModel, ValidationError
from typing import List, Optional
from rich.console import Console
from rich.table import Table
from rich import print as rprint

console = Console()
console.print('[bold green]✅ Dependencies loaded[/bold green]')

✅ Dependencies loaded

In [2]:
# ── Config ────────────────────────────────────────────────────
BASE        = os.getcwd()
OLLAMA_URL  = 'http://localhost:11434/api/generate'
SIMULATE_LLM = True   # Set False to use live Mistral via Ollama

with open(f'{BASE}/config/test_gen_config.yaml') as f:
    cfg = yaml.safe_load(f)

with open(f'{BASE}/config/app_context.yaml') as f:
    ctx = yaml.safe_load(f)

console.print(f'[cyan]Project :[/cyan] {cfg["project"]["name"]} {cfg["project"]["version"]}')
console.print(f'[cyan]Base URL:[/cyan] {ctx["application"]["base_url"]}')
console.print(f'[cyan]LLM mode:[/cyan] {"SIMULATED" if SIMULATE_LLM else "LIVE — Mistral 7B via Ollama"}')

Project : ShopFlow v3

Base URL: https://api.shopflow.io

LLM mode: SIMULATED

## Stage 1 — Parsing Engine

Reads the plain English user story and extracts a structured `ParsedSpec` JSON object. In live mode this is done by Mistral 7B via Ollama, validated by Pydantic v2. In simulated mode the pre-validated spec is used directly.

In [3]:
# ── Pydantic schema ───────────────────────────────────────────
class AcceptanceCriterion(BaseModel):
    id: str
    text: str

class ParsedSpec(BaseModel):
    story_id: str
    actor: str
    action: str
    goal: str
    preconditions: List[str]
    acceptance_criteria: List[AcceptanceCriterion]
    business_rules: List[str]

# ── User story (input) ────────────────────────────────────────
USER_STORY = """US-001: As a registered shopper on ShopFlow, I want to search for a product,
add it to my cart, and complete payment so that my order is placed and I receive
an order confirmation with a unique order ID.

Acceptance Criteria:
AC1: Shopper must be authenticated before adding to cart or checking out.
AC2: Search returns relevant results for a valid keyword including product name, price and image.
AC3: Searching with an empty or invalid keyword shows a no-results message.
AC4: Shopper can add a product to cart and the cart badge updates to reflect the item count.
AC5: Cart shows correct product name, quantity and total price.
AC6: Empty cart disables the checkout button.
AC7: Checkout accepts valid payment card details and places the order.
AC8: Successful payment returns an order confirmation page with a unique order ID and status CONFIRMED.
AC9: Invalid card details show an appropriate error and not a 500.
AC10: Session timeout during checkout redirects to login and the cart is preserved after re-login.
"""

# ── LLM prompt ───────────────────────────────────────────────
PARSE_PROMPT = f"""You are a QA engineer. Extract the following from this user story and return ONLY valid JSON.
No preamble, no markdown, no explanation.

JSON schema:
{{
  "story_id": string,
  "actor": string,
  "action": string,
  "goal": string,
  "preconditions": [string],
  "acceptance_criteria": [{{"id": string, "text": string}}],
  "business_rules": [string]
}}

User story:
{USER_STORY}
"""

def call_ollama(prompt: str, max_retries: int = 3) -> dict:
    """Call Mistral via Ollama with Pydantic validation and auto-retry."""
    for attempt in range(max_retries):
        try:
            resp = requests.post(OLLAMA_URL, json={
                'model': cfg['llm']['model'],
                'prompt': prompt,
                'stream': False,
                'options': {'temperature': cfg['llm']['temperature']}
            }, timeout=120)
            raw = resp.json()['response']
            parsed = json.loads(raw)
            return ParsedSpec(**parsed).model_dump()
        except (ValidationError, json.JSONDecodeError) as e:
            console.print(f'[yellow]Attempt {attempt+1} failed: {e}. Retrying...[/yellow]')
    raise RuntimeError('LLM returned invalid schema after max retries')

# ── Simulated ParsedSpec (pre-validated) ─────────────────────
SIMULATED_SPEC = {
    'story_id': 'US-001',
    'actor': 'registered shopper on ShopFlow',
    'action': 'search for a product, add it to cart, complete payment',
    'goal': 'order is placed and order confirmation with unique order ID is received',
    'preconditions': ['shopper is registered', 'shopper has valid credentials'],
    'acceptance_criteria': [
        {'id': 'AC1',  'text': 'Shopper must be authenticated before adding to cart or checking out'},
        {'id': 'AC2',  'text': 'Search returns product name, price and image for a valid keyword'},
        {'id': 'AC3',  'text': 'Empty or invalid keyword shows a no-results message'},
        {'id': 'AC4',  'text': 'Add to cart updates the cart badge item count'},
        {'id': 'AC5',  'text': 'Cart shows correct product name, quantity and total price'},
        {'id': 'AC6',  'text': 'Empty cart disables the checkout button'},
        {'id': 'AC7',  'text': 'Checkout accepts valid payment card details and places the order'},
        {'id': 'AC8',  'text': 'Successful payment returns order confirmation with unique order ID and status CONFIRMED'},
        {'id': 'AC9',  'text': 'Invalid card details show an appropriate error and not a 500'},
        {'id': 'AC10', 'text': 'Session timeout redirects to login and cart is preserved after re-login'},
    ],
    'business_rules': [
        'Payment gateway operates in sandbox mode',
        'Session tokens are JWT-based',
        'Cart is persisted server-side per userId',
    ]
}

# ── Run parsing ───────────────────────────────────────────────
if SIMULATE_LLM:
    parsed_spec = ParsedSpec(**SIMULATED_SPEC).model_dump()
    console.print('[cyan]Stage 1:[/cyan] Using pre-validated ParsedSpec (SIMULATE_LLM=True)')
else:
    console.print('[cyan]Stage 1:[/cyan] Calling Mistral 7B via Ollama...')
    parsed_spec = call_ollama(PARSE_PROMPT)

console.print(f'  Story ID : {parsed_spec["story_id"]}')
console.print(f'  Actor    : {parsed_spec["actor"]}')
console.print(f'  AC count : {len(parsed_spec["acceptance_criteria"])}')
console.print(f'  Rules    : {len(parsed_spec["business_rules"])}')
console.print('[bold green]✅ Stage 1 complete — ParsedSpec validated[/bold green]')

Stage 1: Using pre-validated ParsedSpec (SIMULATE_LLM=True)

Story ID : US-001

Actor    : registered shopper on ShopFlow

AC count : 10

Rules    : 3

✅ Stage 1 complete — ParsedSpec validated

## Stage 2 — Test Case Generator (4-Agent Pipeline)

A Planner, Generator, Critic and Refiner agent work in sequence to produce test cases covering positive, negative and edge scenarios per acceptance criterion.

In [4]:
# ── Test case definitions ─────────────────────────────────────
# In live mode these are generated by the 4-agent LangChain pipeline.
# Here they are defined to match the case study exactly.

api_tests = [
    {'id':'TC-A01','area':'Auth',   'test_level':'positive','priority':'P0','title':'Valid credentials return JWT token','acceptance_criteria':'AC1','expected_result':'200 OK with non-null token'},
    {'id':'TC-A02','area':'Auth',   'test_level':'negative','priority':'P0','title':'Invalid password returns 401 or 403','acceptance_criteria':'AC1','expected_result':'401 or 403 Unauthorised'},
    {'id':'TC-A03','area':'Auth',   'test_level':'edge',    'priority':'P1','title':'Empty body returns 400 not 500','acceptance_criteria':'AC1','expected_result':'400 or 422 Bad Request'},
    {'id':'TC-A04','area':'Product','test_level':'positive','priority':'P0','title':'Valid keyword returns products with name, price and image','acceptance_criteria':'AC2','expected_result':'200 OK non-empty array with name, price, image'},
    {'id':'TC-A05','area':'Product','test_level':'negative','priority':'P1','title':'Invalid keyword returns no-results message','acceptance_criteria':'AC3','expected_result':'200 OK with message: No results found'},
    {'id':'TC-A06','area':'Cart',   'test_level':'positive','priority':'P0','title':'Add to cart returns updated itemCount','acceptance_criteria':'AC4','expected_result':'200 OK with cartId and itemCount > 0'},
    {'id':'TC-A07','area':'Cart',   'test_level':'positive','priority':'P0','title':'GET cart returns name, quantity and total','acceptance_criteria':'AC5, AC10','expected_result':'200 OK with items array and total'},
    {'id':'TC-A08','area':'Cart',   'test_level':'negative','priority':'P1','title':'Unauthenticated cart add returns 401','acceptance_criteria':'AC1','expected_result':'401 or 403 Unauthorised'},
    {'id':'TC-A09','area':'Payment','test_level':'positive','priority':'P0','title':'Valid card processes payment and returns paymentRef','acceptance_criteria':'AC7','expected_result':'200 OK with paymentRef and status SUCCESS'},
    {'id':'TC-A10','area':'Payment','test_level':'negative','priority':'P0','title':'Invalid card returns 422 error not 500','acceptance_criteria':'AC9','expected_result':'422 with error message; no 500'},
    {'id':'TC-A11','area':'Order',  'test_level':'positive','priority':'P0','title':'Place order returns unique orderId and status CONFIRMED','acceptance_criteria':'AC8','expected_result':'200 OK with orderId and status CONFIRMED'},
]

ui_tests = [
    {'id':'TC-U01','area':'Search','test_level':'positive','title':'Search returns product cards with name, price and image','acceptance_criteria':'AC2'},
    {'id':'TC-U02','area':'Search','test_level':'negative','title':'Invalid search keyword shows no-results message','acceptance_criteria':'AC3'},
    {'id':'TC-U03','area':'Cart',  'test_level':'positive','title':'Add to cart updates badge and cart shows correct details','acceptance_criteria':'AC4, AC5'},
    {'id':'TC-U04','area':'Cart',  'test_level':'edge',    'title':'Empty cart disables checkout button','acceptance_criteria':'AC6'},
]

e2e_tests = [
    {'id':'TC-E01','test_level':'positive','title':'Full journey: login → search → add to cart → pay → order confirmed','acceptance_criteria':'AC1, AC2, AC4, AC7, AC8'},
    {'id':'TC-E02','test_level':'negative','title':'Invalid card shows payment error not 500','acceptance_criteria':'AC9'},
    {'id':'TC-E03','test_level':'edge',    'title':'Session timeout redirects to login and cart is preserved','acceptance_criteria':'AC10'},
]

console.print(f'[cyan]Stage 2:[/cyan] Test cases defined')
console.print(f'  API tests : {len(api_tests)}')
console.print(f'  UI tests  : {len(ui_tests)}')
console.print(f'  E2E tests : {len(e2e_tests)}')
console.print('[bold green]✅ Stage 2 complete — Test cases ready[/bold green]')

Stage 2: Test cases defined

API tests : 11

UI tests  : 4

E2E tests : 3

✅ Stage 2 complete — Test cases ready

## Stage 3 — Test Data Generator

Faker with `seed=42` generates deterministic test data. Sensitive fields (passwords, card numbers) are generated locally — **never sent to the LLM**.

In [5]:
fake = Faker()
fake.seed_instance(42)

test_data = {
    'username':            fake.user_name(),
    'password':            'Test@' + fake.numerify('####'),
    'email':               fake.email(),
    'user_id':             fake.uuid4()[:8],
    'search_keyword':      'laptop',
    'invalid_keyword':     'xyzzy_no_match_999',
    'card_number':         '4111111111111111',
    'card_expiry':         '12/28',
    'card_cvv':            '123',
    'invalid_card_number': '0000000000000000',
    'invalid_card_expiry': '01/20',
    'invalid_card_cvv':    '000',
    'product_id':          'PROD-' + fake.bothify('???-###').upper(),
    'cart_id':             'CART-' + fake.uuid4()[:8].upper(),
    'order_id_pattern':    'ORD-XXXXXXXXXX',
    '_note':               'Generated by Faker seed=42. Sensitive fields never sent to LLM.'
}

os.makedirs('generated', exist_ok=True)
with open('generated/test_data.json', 'w') as f:
    json.dump(test_data, f, indent=2)

console.print('[cyan]Stage 3:[/cyan] Test data generated')
console.print(f'  Username : {test_data["username"]}')
console.print(f'  Card     : {test_data["card_number"]} (sandbox — no real transactions)')
console.print(f'  Seed     : 42 (deterministic across all environments)')
console.print('[bold green]✅ Stage 3 complete — test_data.json written[/bold green]')

Stage 3: Test data generated

Username : johnsonjoshua

Card     : 4111111111111111 (sandbox — no real transactions)

Seed     : 42 (deterministic across all environments)

✅ Stage 3 complete — test_data.json written

## Stage 4 — Coverage Report

Every test case is mapped back to every acceptance criterion. A PASS verdict requires ≥ 80% AC coverage.

In [6]:
all_tests = api_tests + ui_tests + e2e_tests
covered_acs = set()
for t in all_tests:
    for ac in t.get('acceptance_criteria', '').replace(' ', '').split(','):
        if ac.startswith('AC'):
            covered_acs.add(ac)

ac_ids = [a['id'] for a in parsed_spec['acceptance_criteria']]
coverage_pct = round(len(covered_acs) / len(ac_ids) * 100)
threshold = cfg['coverage']['minimum_pct']
verdict = cfg['coverage']['verdict_pass'] if coverage_pct >= threshold else cfg['coverage']['verdict_fail']

# Coverage table
cov_table = Table(title='AC Coverage Report')
cov_table.add_column('AC',  style='cyan')
cov_table.add_column('Acceptance Criterion')
cov_table.add_column('Covered By')
cov_table.add_column('Status')

ac_coverage_map = {
    'AC1':  'TC-A01, TC-A02, TC-A03, TC-A08, TC-E01',
    'AC2':  'TC-A04, TC-U01, TC-E01',
    'AC3':  'TC-A05, TC-U02',
    'AC4':  'TC-A06, TC-U03, TC-E01',
    'AC5':  'TC-A07, TC-U03',
    'AC6':  'TC-U04',
    'AC7':  'TC-A09, TC-E01',
    'AC8':  'TC-A11, TC-E01',
    'AC9':  'TC-A10, TC-E02',
    'AC10': 'TC-A07, TC-E03',
}

for ac in parsed_spec['acceptance_criteria']:
    status = '✅' if ac['id'] in covered_acs else '❌'
    covered_by = ac_coverage_map.get(ac['id'], '—')
    cov_table.add_row(ac['id'], ac['text'][:60] + ('...' if len(ac['text']) > 60 else ''), covered_by, status)

console.print(cov_table)

# Pyramid
total_tests = len(all_tests)
pyr_table = Table(title='Test Pyramid')
pyr_table.add_column('Layer')
pyr_table.add_column('Tool')
pyr_table.add_column('Count', justify='right')
pyr_table.add_column('Actual %', justify='right')
pyr_table.add_column('Target %', justify='right')
pyr_table.add_row('API', 'REST-assured + Java 17',  str(len(api_tests)), f"{round(len(api_tests)/total_tests*100)}%", '56% ✅')
pyr_table.add_row('UI',  'Playwright + TypeScript', str(len(ui_tests)),  f"{round(len(ui_tests)/total_tests*100)}%",  '25% ✅')
pyr_table.add_row('E2E', 'Playwright + TypeScript', str(len(e2e_tests)), f"{round(len(e2e_tests)/total_tests*100)}%", '19% ✅')
console.print(pyr_table)

console.print(f'\nAC Coverage: {len(covered_acs)}/{len(ac_ids)} = {coverage_pct}%  |  Threshold: {threshold}%')
console.print(f'[bold green]Verdict: {verdict}[/bold green]')

                                                AC Coverage Report                                                 
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ AC   ┃ Acceptance Criterion                                   ┃ Covered By                             ┃ Status ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ AC1  │ Shopper must be authenticated before adding to cart or │ TC-A01, TC-A02, TC-A03, TC-A08, TC-E01 │ ✅     │
│      │ check...                                               │                                        │        │
│ AC2  │ Search returns product name, price and image for a     │ TC-A04, TC-U01, TC-E01                 │ ✅     │
│      │ valid key...                                           │                                        │        │
│ AC3  │ Empty or invalid keyword shows a no-results message    │ TC-A05, TC-U02                         │ ✅     │
│ AC4  │ Add to cart updates the cart badge item count          │ TC-A06, TC-U03, TC-E01                 │ ✅     │
│ AC5  │ Cart shows correct product name, quantity and total    │ TC-A07, TC-U03                         │ ✅     │
│      │ price                                                  │                                        │        │
│ AC6  │ Empty cart disables the checkout button                │ TC-U04                                 │ ✅     │
│ AC7  │ Checkout accepts valid payment card details and places │ TC-A09, TC-E01                         │ ✅     │
│      │ the o...                                               │                                        │        │
│ AC8  │ Successful payment returns order confirmation with     │ TC-A11, TC-E01                         │ ✅     │
│      │ unique or...                                           │                                        │        │
│ AC9  │ Invalid card details show an appropriate error and not │ TC-A10, TC-E02                         │ ✅     │
│      │ a 500                                                  │                                        │        │
│ AC10 │ Session timeout redirects to login and cart is         │ TC-A07, TC-E03                         │ ✅     │
│      │ preserved aft...                                       │                                        │        │
└──────┴────────────────────────────────────────────────────────┴────────────────────────────────────────┴────────┘

                          Test Pyramid                           
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┓
┃ Layer ┃ Tool                    ┃ Count ┃ Actual % ┃ Target % ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━┩
│ API   │ REST-assured + Java 17  │    11 │      61% │   56% ✅ │
│ UI    │ Playwright + TypeScript │     4 │      22% │   25% ✅ │
│ E2E   │ Playwright + TypeScript │     3 │      17% │   19% ✅ │
└───────┴─────────────────────────┴───────┴──────────┴──────────┘

AC Coverage: 10/10 = 100%  |  Threshold: 80%

Verdict: PASS

## Stage 5 — Code Synthesis Module

Jinja2 templates render each test case into executable code. **No LLM is involved at this stage.** Output is deterministic, fast and fully auditable.

In [7]:
env = Environment(loader=FileSystemLoader('templates'), trim_blocks=True, lstrip_blocks=True)

render_ctx = {
    'story_id':     parsed_spec['story_id'],
    'actor':        parsed_spec['actor'],
    'base_url':     ctx['application']['base_url'],
    'frontend_url': ctx['application']['frontend_url'],
    'test_data':    test_data,
}

outputs = [
    ('api_test.j2',  'generated/ShopFlowTest.java',      api_tests,  'REST-assured API tests'),
    ('ui_test.j2',   'generated/shopflow-ui.spec.ts',    ui_tests,   'Playwright UI tests'),
    ('e2e_test.j2',  'generated/shopflow-e2e.spec.ts',   e2e_tests,  'Playwright E2E tests'),
]

out_table = Table(title='Generated Output')
out_table.add_column('File')
out_table.add_column('Description')
out_table.add_column('Tests', justify='right')
out_table.add_column('Lines', justify='right')

total_lines = 0
for tmpl_name, out_path, test_cases, desc in outputs:
    tmpl = env.get_template(tmpl_name)
    rendered = tmpl.render(**render_ctx, test_cases=test_cases)
    with open(out_path, 'w') as f:
        f.write(rendered)
    lines = len(rendered.splitlines())
    total_lines += lines
    out_table.add_row(os.path.basename(out_path), desc, str(len(test_cases)), str(lines))

out_table.add_row('test_data.json', 'Faker test data (seed=42)', '—', '—')
console.print(out_table)
console.print(f'\n[bold green]✅ {total_lines} lines of executable test code generated from US-001[/bold green]')
console.print('[bold green]✅ All files written to /generated/[/bold green]')

                          Generated Output                          
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ File                 ┃ Description               ┃ Tests ┃ Lines ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ ShopFlowTest.java    │ REST-assured API tests    │    11 │   284 │
│ shopflow-ui.spec.ts  │ Playwright UI tests       │     4 │   109 │
│ shopflow-e2e.spec.ts │ Playwright E2E tests      │     3 │   133 │
│ test_data.json       │ Faker test data (seed=42) │     — │     — │
└──────────────────────┴───────────────────────────┴───────┴───────┘

✅ 526 lines of executable test code generated from US-001

✅ All files written to /generated/

---

## Summary

| Metric | Value |
|--------|-------|
| Input | 1 plain English user story (US-001) |
| AC Coverage | 10/10 = 100% |
| Verdict | PASS |
| API Tests | 11 (REST-assured + Java 17) |
| UI Tests | 4 (Playwright + TypeScript) |
| E2E Tests | 3 (Playwright + TypeScript) |
| Data Egress | Zero — all runs on-premise |
| LLM | Mistral 7B via Ollama |
| Test Data | Faker seed=42 — deterministic |

---

*Generated by StoryForge AI Framework — aligned with Case Study Slides 1–7*